# Dataset Creation and Curation

This notebook summarizes how the PPE dataset for the **Smart Factory Safety Inspector** project was built and curated before training. It is a result notebook, not an executable analysis notebook.

The primary narrative source is `../docs/Dataset_Building_Report.md`, supported by Dataset Builder reports and final project EDA artifacts. The purpose is to show that the dataset was a major engineering contribution, not simply a downloaded YOLO dataset.

![Dataset creation pipeline](../outputs/dataset_creation/figures/dataset_creation_pipeline.png)


## 1. Dataset Creation Overview

The dataset-building process combined broad raw data collection, automated cleaning, manual annotation review in CVAT, synthetic supplementation, final consolidation, leakage control, and later project-level QA cleanup.

| Stage | Evidence-supported result |
|---|---:|
| Initial supported raw images audited | 361,939 |
| Redundant exact duplicate files identified in raw audit | 44,903 |
| Source variants rejected during selection audit | 146,316 |
| Images manually reviewed in CVAT or connected review workflows | about 13,000 |
| Dataset-building snapshot images | 3,882 |
| Dataset-building snapshot annotations | 23,772 |
| Current canonical project images after later QA cleanup | 3,865 |
| Current canonical project annotations after later QA cleanup | 23,686 |

The builder snapshot and the current canonical project dataset are intentionally distinguished throughout this notebook.


## 2. Why a Custom Dataset Was Necessary

Public PPE datasets were useful raw material, but the project required a stable five-class worker-centered dataset. A direct merge would have preserved class mismatches, annotation gaps, duplicate leakage risk, and off-domain images.

| Problem found in raw sources | Why it mattered for this project |
|---|---|
| Different class definitions and class names | The model needed one stable schema: person, helmet, vest, gloves, goggles. |
| Missing person boxes | Compliance is evaluated per worker, so PPE boxes need worker context. |
| Missing PPE boxes | Visible PPE without labels teaches the model that PPE can be background. |
| Incorrect boxes or wrong classes | Poor ground truth can be more damaging than a smaller clean dataset. |
| Unnecessary classes | Extra classes would distract from the final safety-inspection task. |
| Boots and ordinary footwear mixed together | The raw evidence was not reliable enough for a clean safety-boot class. |
| Helmet-heavy bias | Vests, gloves, and goggles needed deliberate coverage and review. |
| Product crops and off-domain images | The final model needed worker scenes, not isolated objects or unrelated imagery. |
| Duplicates and source variants | Repeated images can inflate dataset size and create split leakage. |
| Small PPE objects | Gloves and goggles often required human judgment during annotation review. |


## 3. Target Dataset Definition

The final detection task was defined before consolidation so that raw sources, model-assisted labels, and manual CVAT review could be normalized into one schema.

| Class ID | Class name | Dataset role |
|---:|---|---|
| 0 | person | Worker instance used for compliance association. |
| 1 | helmet | Head protection object. |
| 2 | vest | Safety vest or high-visibility vest object. |
| 3 | gloves | Visible glove object; multiple glove boxes per worker can be valid. |
| 4 | goggles | Clearly visible eyewear under the project definition, including safety goggles, safety glasses, prescription glasses, sunglasses, clear glasses, and tinted glasses when visible enough to annotate. |

Important schema decisions:

| Decision | Rationale |
|---|---|
| Include person | The application makes worker-level compliance decisions. |
| Use object-detection boxes only | Each YOLO label row represents one visible object. |
| Exclude boots | Source data mixed safety boots with ordinary shoes, sandals, slippers, and feet-only crops. |
| Keep five classes | The final dataset stays focused on the PPE requirements used by the application. |


## 4. Raw Data Collection

The Dataset Builder records preserve source data mainly as consolidated raw pools and source-family groups. The notebook therefore uses audited source-family descriptions instead of inventing external dataset names that are not present in the frozen records.

| Source area or family | Main contribution | Main issue requiring curation |
|---|---|---|
| `data/Raw_images` consolidated public/raw image pool | Broad worker and PPE coverage | Required duplicate removal, source-variant control, quality checks, and manual filtering. |
| Roboflow-style mixed source families | Helmet, vest, gloves, goggles, and person examples | Mixed provenance and repeated `.rf...` style variants required source-family audits. |
| Worker/construction families such as `hard_hat_workers`, `ppe`, `dataset1`-`dataset6`, `alet_wikimgs_construction` | Realistic worker and PPE scenes | Some families were coherent, while others were broad catch-all groups. |
| Off-domain and off-target families | Mostly rejected | Included ordinary mask imagery, footwear-focused material, composite images, or scenes not useful for worker PPE detection. |
| Synthetic Data 01 and Synthetic Data 02 | Controlled PPE combinations and difficult cases | Needed manual review before inclusion and was restricted to train only. |

![Audited dataset-building scale](../outputs/dataset_creation/figures/dataset_builder_audit_scale.png)


## 5. Automated Cleaning and Standardization

Automated cleaning narrowed the raw pool into a safer review workflow. These operations did not replace manual annotation review; they removed obvious structural, quality, and provenance risks before CVAT work.

| Operation | Purpose | Result type |
|---|---|---|
| Class remapping | Normalize raw labels to the five-class target schema. | Standardized labels. |
| Unwanted-class filtering | Remove classes outside the final task. | Cleaner candidate pool. |
| Exact duplicate detection by SHA256 | Detect identical file content. | Duplicate evidence and exclusion plans. |
| Source-variant control | Reduce repeated near-identical source variants. | Lower leakage and memorization risk. |
| Image readability and resolution checks | Remove unreadable or unsuitable files. | Safer physical dataset. |
| YOLO label validation | Catch malformed labels and image-label pairing problems. | Validated annotation files. |
| Filename and path-safety checks | Prevent broken or unsafe materialization paths. | Reproducible dataset layout. |
| Quality audits | Flag very small, blurry, dark, bright, low-contrast, low-information, or extreme-aspect images. | Review prioritization. |
| Family-level review | Identify repetitive, off-domain, footwear, and composite-image families. | Better source filtering. |
| Candidate balancing | Avoid letting a few source families dominate review. | More useful manual review pool. |

The selection audit rejected **44,903 exact duplicate files** and **146,316 source variants**. Auto-annotation helped accelerate review, but it was not treated as final ground truth without human inspection.


## 6. Large-Scale Manual Review in CVAT

Manual CVAT review was a central dataset-building stage. The supported record is **about 13,000 images manually reviewed in CVAT or connected manual review workflows**. This number should be read carefully: it includes manual inspection and review workflows, not a claim that every image was annotated from a blank canvas.

| Manual review activity | What it meant |
|---|---|
| Visual inspection | Check whether each candidate image was useful and annotatable for the target task. |
| Annotation correction | Adjust boxes, remove false labels, correct wrong classes, and add missing labels where needed. |
| Worker-context checking | Ensure PPE could be interpreted relative to visible workers. |
| Small-object review | Pay special attention to gloves and goggles, which are often tiny or partially visible. |
| Filtering | Reject off-domain, unsafe, ambiguous, repetitive, or low-value images. |
| Post-export validation | Recheck exported YOLO labels and file integrity after review. |

| Scope item | Supported value or limitation |
|---|---|
| Manual review scale | About 13,000 images manually reviewed in CVAT or connected review workflows. |
| Confirmed modified-frame count | 1,813 frames modified across Batch 01 through Batch 05. |
| Later batch records | Batch 06 through Batch 11 preserve reviewed final-batch retention/removal outcomes, but the source report does not provide equivalent modified-frame counts. |
| CVAT screenshot evidence | No actual CVAT interface screenshots are embedded here; manual-review evidence is represented through audited batch records, retained/removed counts, and final annotated examples. |
| Ground-truth policy | Existing or model-generated YOLO annotations were visually inspected and corrected; auto-labels alone were not accepted as final ground truth. |

The practical point is that the dataset was curated by repeated human-in-the-loop review, not just by automated scripts.


## 7. Review Batches and Dataset Filtering

The real-image pool came from 11 reviewed batches. Many candidates were useful as search results or model-assisted proposals, but not reliable enough for the final training dataset.

| Batch | Candidates | Retained | Removed | Notes |
|---|---:|---:|---:|---|
| Batch 01 | 1,000 | 568 | 432 | 474 frames modified during review. |
| Batch 02 | 1,000 | 524 | 476 | 403 frames modified during review. |
| Batch 03 | 1,000 | 455 | 545 | 354 frames modified during review. |
| Batch 04 | 1,000 | 363 | 637 | 305 frames modified during review. |
| Batch 05 | 1,000 | 328 | 672 | 277 frames modified during review. |
| Batch 06 | 845 | 181 | 664 | Reviewed final batch. |
| Batch 07 | 845 | 183 | 662 | Reviewed final batch. |
| Batch 08 | 845 | 212 | 633 | Reviewed final batch. |
| Batch 09 | 845 | 205 | 640 | Reviewed final batch. |
| Batch 10 | 845 | 153 | 692 | Reviewed final batch. |
| Batch 11 | 845 | 104 | 741 | Reviewed final batch. |
| **Total** | **10,070** | **3,276** | **6,794** | Retained review-batch records before later consolidation exclusions. |

![Manual review batch outcomes](../outputs/dataset_creation/figures/manual_review_batch_outcomes.png)

This filtering rate is important evidence: the final dataset was intentionally smaller than the candidate pool because weak or unsafe examples were rejected.


## 8. Synthetic Data Supplementation

Synthetic data was used to supplement difficult PPE combinations and underrepresented visual cases. It was not accepted automatically, and it was kept out of validation and test.

| Synthetic source | Candidates | Retained | Removed | Final use |
|---|---:|---:|---:|---|
| Synthetic Data 01 | 500 | 336 | 164 | Train only |
| Synthetic Data 02 | 311 | 305 | 6 | Train only |
| **Total synthetic reviewed** | **811** | **641** | **170** | **Train only** |

![Synthetic data review contribution](../outputs/dataset_creation/figures/synthetic_data_contribution.png)

Validation and test remained real-only so final model selection and final evaluation were not measured on generated imagery.


## 9. Dataset Consolidation

After review, the retained real and synthetic records were consolidated through an audited release pipeline.

| Consolidation step | Evidence-supported result |
|---|---:|
| Reviewed input sources | 13 |
| Input records before final exclusions | 3,917 |
| Zero-label records excluded | 35 |
| Exact duplicate groups in final candidate pool | 0 |
| Exact duplicate exclusions at final candidate stage | 0 |
| Final retained records in builder snapshot | 3,882 |

The consolidation process used SHA256 hashes, label hashes, source manifests, deterministic filenames, and validation reports to protect against stale paths, duplicate leakage, and incorrect source mappings.

![Dataset-builder split composition](../outputs/dataset_creation/figures/dataset_builder_split_composition.png)


## 10. Duplicate and Leakage Control

Duplicate and leakage control was handled at multiple stages: raw audit, source-variant filtering, final consolidation, and final project EDA after later cleanup.

| Check or control | Dataset-building evidence |
|---|---:|
| Exact duplicate groups in initial raw audit | 37,485 |
| Redundant duplicate files in initial raw audit | 44,903 |
| Source variants rejected during selection audit | 146,316 |
| Exact duplicate groups in final candidate pool | 0 |
| Split-group leakage in builder snapshot | 0 |
| Image-SHA leakage in builder snapshot | 0 |
| Synthetic images in validation | 0 |
| Synthetic images in test | 0 |

The current canonical project dataset was later checked again by EDA and Data Processing. It now has **0 exact duplicate image pairs**, **0 perceptual near-duplicate candidates**, and **0 cross-split leakage candidates**.


## 11. Dataset Evolution

The Dataset Builder snapshot and the current project dataset are related but not identical. The builder snapshot is the curated dataset release described in the source report. The current canonical dataset reflects later final-project QA cleanup documented in the Data Processing notebook.

![Dataset version distinction](../outputs/dataset_creation/figures/dataset_version_distinction.png)

| Metric | Dataset-building snapshot | Current canonical final project dataset |
|---|---:|---:|
| Images | 3,882 | 3,865 |
| Label files | 3,882 | 3,865 |
| Annotations | 23,772 | 23,686 |
| Train images | 3,234 | 3,219 |
| Validation images | 324 | 323 |
| Test images | 324 | 323 |
| Duplicate/leakage status | 0 split leakage | 0 exact duplicates, 0 near-duplicate candidates, 0 cross-split leakage candidates |

The later project-level cleanup removed confirmed duplicate/source variants and repaired confirmed annotation issues. The detailed cleanup evidence belongs to `02_Data_Processing.ipynb`, so it is summarized here only to prevent mixing historical and current dataset counts.


## 12. Final Dataset Statistics

The final project uses the current canonical dataset in `data/final_dataset/`.

| Split | Images | Label files |
|---|---:|---:|
| Train | 3,219 | 3,219 |
| Validation | 323 | 323 |
| Test | 323 | 323 |
| **Total** | **3,865** | **3,865** |

| Class | Current annotations |
|---|---:|
| person | 7,973 |
| helmet | 4,371 |
| vest | 3,219 |
| gloves | 4,999 |
| goggles | 3,124 |
| **Total** | **23,686** |

![Current class counts by split](../outputs/eda/figures/class_counts_by_split.png)

The current class distribution remains broader than a helmet-only PPE dataset. Gloves can legitimately appear more than once per worker, so the high glove count is expected and not automatically a label-quality problem.


## 13. Final Quality Assurance

The current canonical dataset passed the final EDA readiness checks after Data Processing.

| QA check | Current result |
|---|---:|
| Total images | 3,865 |
| Total label files | 3,865 |
| Missing image-label pairs | 0 |
| Unsupported image files | 0 |
| Unreadable images | 0 |
| Empty label files | 0 |
| Malformed or unusable annotations | 0 |
| Invalid class IDs | 0 |
| Duplicate annotation rows | 0 |
| Large boundary overflows | 0 |
| Exact duplicate image pairs | 0 |
| Perceptual near-duplicate candidates | 0 |
| Cross-split leakage candidates | 0 |
| Final EDA readiness verdict | READY FOR TRAINING |

Remaining EDA findings such as tiny floating-point boundary overflows, small PPE boxes, unassigned PPE under a strict center-point association rule, and multiple same-class PPE per person are documented as limitations or review heuristics, not confirmed blocking dataset errors.


## 14. Key Dataset Engineering Contributions

The final dataset reflects several distinct engineering contributions.

| Contribution | Why it mattered |
|---|---|
| Custom task schema | Converted heterogeneous raw data into one worker-centered PPE detection task. |
| Source-family auditing | Reduced off-domain, repetitive, and provenance-risky data. |
| Automated cleaning | Removed structural problems before manual review and preserved reproducibility. |
| CVAT manual review | Corrected labels and rejected unsafe images at a scale automation could not handle alone. |
| Synthetic supplementation | Added reviewed train-only coverage for difficult PPE combinations. |
| Leakage-safe splitting | Protected validation and test from duplicate/source leakage and synthetic contamination. |
| Final EDA and Data Processing QA | Verified that the canonical project dataset was ready for training after confirmed cleanup. |

Representative current ground-truth examples:

![Representative annotation example](../outputs/eda/examples/normal_annotation_01_005888.png)

![Representative small PPE example](../outputs/eda/examples/small_ppe_01_ChatGPT_Image_1_._2026_._17_31_45.png)


## 15. Conclusion

The Smart Factory Safety Inspector dataset was built through a reproducible curation workflow: raw source collection, schema standardization, automated cleaning, large-scale CVAT review, synthetic supplementation, audited consolidation, duplicate and leakage control, and final project-level QA.

The important outcome is not just dataset size. The final dataset is smaller than the raw pool because weak, duplicated, off-domain, ambiguous, or unreliable examples were removed. The current canonical dataset contains **3,865 images**, **23,686 annotations**, **5 classes**, and no confirmed blocking dataset problems.

This makes the dataset itself a substantial engineering artifact and a strong foundation for the later model comparison, final evaluation, and end-to-end compliance checks.
